In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
import math 

class HeadAttention(nn.Module):
    def __init__(self, emb_size: int, head_size: int, max_seq_len: int):
        super().__init__()
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len

        self.W_k = nn.Linear(self.emb_size, self.head_size)
        self.W_q = nn.Linear(self.emb_size, self.head_size)
        self.W_v = nn.Linear(self.emb_size, self.head_size)

        self.mask = torch.tril(torch.ones(self.max_seq_len, self.max_seq_len))

    def forward(self, x: torch.Tensor):
        batch_size, seq_len, emb_size = x.shape
        key = self.W_k(x)
        query = self.W_q(x)
        value = self.W_v(x)

        attention_matrix : torch.Tensor = query @ key.transpose(1,2) / math.sqrt(self.head_size)

        mask = self.mask[:seq_len, :seq_len] == 0
        
        attention_matrix.masked_fill_(mask, float('-inf'))
        attention_matrix = torch.softmax(attention_matrix, 2) 
        
        return attention_matrix @ value
        


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads: int, emb_size: int,
                head_size: int, max_seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.dropout = dropout

        self.head_attentions = nn.ModuleList([HeadAttention(self.emb_size, self.head_size, self.max_seq_len) for i in range(self.num_heads)])
        self.otpt = nn.Linear(self.head_size * self.num_heads, self.emb_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):

        tensors = [ head_attention(x) for head_attention in self.head_attentions]
        tensor = torch.cat(tensors, dim=2)
        tensor = self.otpt(tensor)
        tensor = self.dropout(tensor)
        return tensor

class FeedForward(nn.Module):
    def __init__(self, emb_size: int, dropout: float == 0.1):
        super().__init__()
        self.emb_size = emb_size
        self.dropout = dropout

        self.first_linear = nn.Linear(emb_size, emb_size * 4)
        self.relu = nn.ReLU()
        self.second_linear = nn.Linear(4 * emb_size, emb_size)
        self.drpt = nn.Dropout(self.dropout)

    def forward(self, x: torch.Tensor):
        tensor = self.first_linear(x)
        tensor = self.relu(tensor)
        tensor = self.second_linear(tensor)
        tensor = self.drpt(tensor)
        return tensor

class Decoder(nn.Module):
    def __init__(self, num_heads: int, emb_size: int,
                head_size: int, max_seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.dropout = dropout

        self.multi_head_attention = MultiHeadAttention(self.num_heads, self.emb_size, self.head_size, self.max_seq_len, self.dropout)
        self.feed_forward = FeedForward(self.emb_size, self.dropout)
        self.first_norm = nn.LayerNorm(self.emb_size)
        self.second_norm = nn.LayerNorm(self.emb_size)

    def forward(self, x: torch.Tensor):
        tensor = self.multi_head_attention(x)
        tensor += x
        tensor = self.first_norm(tensor)
        tensor += self.feed_forward(tensor)
        tensor = self.second_norm(tensor)
        return tensor
